# Module 1: The Zero-to-One Agent 🚀

Welcome to the **AI Agent Workshop**.

In this first module, you will build your very first **AI Agent** using the Google Agent Development Kit (ADK).
### What is an Agent?
An agent is not just a chatbot. It is an AI system that can:
1.  **Reason**: Plan a series of steps to solve a problem.
2.  **Act**: Use "Tools" (functions) to interact with the world (calling APIs, querying databases).
3.  **Reflect**: Check its own work and correct mistakes.

### Our Goal
We will build a **Financial Assistant** that can:
-   Look up stock prices (simulated).
-   Provide investment advice based on market conditions.
-   Explain its reasoning clearly.

## 1. Setup and Installation

We need two key libraries:
-   `google-adk`: The framework for building agents.
-   `google-genai`: The client for accessing Gemini models.
-   `python-dotenv`: To securely load our API keys.

In [ ]:
%pip install google-adk google-genai python-dotenv

In [ ]:
from google.genai import Client
import os
from dotenv import load_dotenv

# Load API key from .env.local
# This keeps your credentials secure and out of the notebook code
load_dotenv('../.env.local')

# Verify we have the key
if "GOOGLE_API_KEY" not in os.environ:
    print("❌ GOOGLE_API_KEY not found! Please create a .env.local file in the project root.")
else:
    print("✅ API Key loaded successfully.")

client = Client(api_key=os.environ['GOOGLE_API_KEY'])

## 2. Your First Tool

Agents rely on **Tools** to get accurate data. LLMs (Large Language Models) like Gemini are trained on past data, so they don't know the *current* stock price of JPM.

Let's create a tool for that.

In [ ]:
from typing import Annotated

def get_stock_price(
    ticker: Annotated[str, "The stock ticker symbol (e.g. JPM, AAPL)"]
) -> float:
    """Fetches the current stock price for a given ticker."""
    print(f"   [Tool] Fetching price for {ticker}...")
    
    # Simulation logic (replace with real API in production)
    mock_prices = {
        "JPM": 245.50,
        "AAPL": 180.00,
        "GOOG": 140.00
    }
    return mock_prices.get(ticker.upper(), 100.00)

### Test the Tool
Always verify your python functions work before giving them to an AI.

In [ ]:
print(f"Price of JPM: ${get_stock_price('JPM')}")

## 3. Creating the Agent

Now we assemble the agent using `google-adk`.

Key Components:
1.  **Model**: `gemini-2.5-flash` (Fast, efficient).
2.  **Tools**: Our `get_stock_price` function.
3.  **Instruction**: The system prompt that defines the agent's persona.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import FunctionTool

# 1. Wrap the function as an ADK Tool
stock_tool = FunctionTool(get_stock_price)

# 2. Define the Agent
agent = Agent(
    model="gemini-2.5-flash",
    name="financial_assistant",
    tools=[stock_tool],  # Give it the tool
    instruction=(
        "You are a helpful Financial Assistant for JPMorgan Chase employees. "
        "Use your tools to find stock prices when asked. "
        "Always explain your analysis in a professional tone."
    )
)

## 4. Running the Agent (Interactive Chat)

We use the `Runner` to manage the conversation session.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types
import uuid

# Setup the runner with in-memory storage (good for testing)
runner = Runner(
    agent=agent,
    app_name="module_1_playground", 
    session_service=InMemorySessionService(),
    auto_create_session=True
)

# Helper function to run a chat loop
async def chat_loop():
    user_id = "user_1"
    session_id = str(uuid.uuid4())
    
    print("🤖 Financial Assistant Ready! (Type 'quit' to exit)")
    print("--------------------------------------------------")

    while True:
        user_input = input("You: ")
        if user_input.lower() in ["quit", "exit"]: 
            break
        
        # Send message to agent
        message = types.Content(role="user", parts=[types.Part(text=user_input)])
        
        # Stream the response
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session_id,
            new_message=message
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(f"Agent: {part.text}")

# Run it!
# await chat_loop()

## 5. Experimentation

Try running the cell above and asking:
1.  "What is the price of JPM?"
    *   *Observation: It should call the tool and give you $245.50.*
2.  "Compare JPM and AAPL stock."
    *   *Observation: It should call the tool TWICE (once for each).* 
3.  "What is the weather in New York?"
    *   *Observation: It should say it doesn't know, because it has no weather tool.*

## 6. Going Production: Understanding Structure

**Learning vs. Deploying**

*   **This Notebook**: Great for experimenting, validiating logic, and learning concepts.
*   **The CLI App**: Required for deployment (e.g., using `adk run` or `adk web`).

During this workshop, we will focus on notebooks for learning. But at the end of each module, check the companion folders (like `module_01/financial_agent_cli/`) to see how this same code is structured for a production app.

**Key Differences in Production:**
1.  Code is in `.py` files, not notebooks.
2.  The main variable must be named `root_agent`.
3.  Dependencies are managed in `requirements.txt` or `pyproject.toml`.

In [ ]:
# Optional: Inspect the production folder structure
!ls -R financial_agent_cli